# Advanced 06 lab — Multimodal adaptation and continual learning

**Scenario.** A small image–text inspection system is validated at legacy Site A. Site B introduces a new camera transform and terminology workflow. Site C is a stronger, untouched reporting source. We must improve the new domain without silently damaging legacy classification, bidirectional retrieval, calibration, or representation geometry.

**Safety and evidence boundary.** This credential-free notebook uses synthetic vectors and a tiny local dual encoder. It is **not a foundation model, VLM benchmark, or production adaptation result**. Site B selects methods and gates. Site C is reporting-only. Training produces candidates; trusted application code validates lineage and decides only `promote_to_shadow`, `needs_review`, `reject`, or `rollback`. No model output authorizes production.

![A validated base moves through bounded adaptation and an independent release gate.](assets/adaptation-lifecycle.svg)


## 1. Reproducible environment and common libraries

The default path uses PyTorch for models and optimization, NumPy/pandas for evidence, scikit-learn for metrics, and Matplotlib for diagnostics. No hidden local module or `lab.py` is imported. Optional PEFT, Transformers, Avalanche, and MLflow integrations remain disabled and revision governed.


In [ ]:
from __future__ import annotations

import copy
import hashlib
import json
import math
import os
import platform
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score

SEED = 20260920
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)

DEVICE = torch.device("cpu")
ARTIFACT_DIR = Path.cwd() / ".artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

CV_ENABLE_PEFT = False
CV_ENABLE_TRANSFORMERS = False
CV_ENABLE_AVALANCHE = False
CV_ENABLE_MLFLOW = False

OPTIONAL_TOOL_MANIFESTS = {
    "peft": {"enabled": CV_ENABLE_PEFT, "revision": "50a277e7c87db460ef7444055788f9da29f2da71"},
    "transformers": {"enabled": CV_ENABLE_TRANSFORMERS, "revision": "c587bc884db2c2e31fc2b8102314656b17aa07b1", "trust_remote_code": False},
    "avalanche": {"enabled": CV_ENABLE_AVALANCHE, "revision": "eb075be393e1f458b2c352514ff6c17b5a2c0f4e"},
    "mlflow": {"enabled": CV_ENABLE_MLFLOW, "revision": "1dfa0fa5b82a600de5cfb80210c12fb60f34f982"},
}
assert not any(item["enabled"] for item in OPTIONAL_TOOL_MANIFESTS.values())

environment = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "device": str(DEVICE),
    "seed": SEED,
    "credential_free": True,
}
environment


## 2. Typed contracts before training

A capability baseline has meaning only when the model, processor, suite, source roles, and metric directions are fixed. Replay data and candidate adapters are governed training artifacts, not anonymous arrays.


In [ ]:
@dataclass(frozen=True)
class BaseModelContract:
    model_id: str
    revision: str
    processor_revision: str
    weights_hash: str
    capability_suite_version: str
    training_scope_known: bool
    license: str


@dataclass(frozen=True)
class ShiftContract:
    shift_id: str
    source_site: str
    target_site: str
    visual_change: str
    language_change: str
    policy_change: str
    evaluation_role: Literal["legacy", "development_only", "reporting_only_no_changes"]


@dataclass(frozen=True)
class CapabilitySuiteContract:
    version: str
    higher_is_better: tuple[str, ...]
    lower_is_better: tuple[str, ...]
    required_slices: tuple[str, ...]


@dataclass(frozen=True)
class ReplayBufferContract:
    buffer_version: str
    selection_method: str
    capacity: int
    source_domains: tuple[str, ...]
    content_policy: str
    digest: str


@dataclass(frozen=True)
class CandidateArtifact:
    candidate_id: str
    base_weights_hash: str
    candidate_weights_hash: str
    method: str
    trainable_parameter_names: tuple[str, ...]
    adapter_config: dict[str, Any]
    training_manifest_hash: str
    replay_buffer_digest: str | None
    capability_suite_version: str
    frozen_policy_hash: str


@dataclass(frozen=True)
class MetricSpec:
    name: str
    direction: Literal["higher_is_better", "lower_is_better"]


@dataclass(frozen=True)
class GateCheck:
    name: str
    status: Literal["PASS", "FAIL", "MISSING"]
    detail: str


@dataclass(frozen=True)
class PromotionDecision:
    candidate_id: str
    decision: Literal["promote_to_shadow", "needs_review", "reject", "rollback"]
    checks: tuple[GateCheck, ...]
    rollback_target_hash: str | None
    authorization: Literal["none"] = "none"


def canonical_hash(value: Any) -> str:
    payload = json.dumps(value, sort_keys=True, separators=(",", ":"), default=str).encode()
    return hashlib.sha256(payload).hexdigest()


CAPABILITY_SUITE = CapabilitySuiteContract(
    version="advanced06-suite-v1",
    higher_is_better=("accuracy", "macro_f1", "image_to_text_accuracy", "text_to_image_accuracy", "paired_cosine"),
    lower_is_better=("ece", "train_time_s", "artifact_bytes"),
    required_slices=("Site A legacy", "Site B development only", "Site C reporting only"),
)

METRIC_SPECS = {
    "macro_f1": MetricSpec("macro_f1", "higher_is_better"),
    "ece": MetricSpec("ece", "lower_is_better"),
    "latency_ms": MetricSpec("latency_ms", "lower_is_better"),
}


def signed_improvement(current: float, reference: float, metric_spec: MetricSpec) -> float:
    '''Positive always means improvement; negative always means regression.'''
    if not (math.isfinite(current) and math.isfinite(reference)):
        raise ValueError("metric_values_must_be_finite")
    if metric_spec.direction == "higher_is_better":
        return float(current - reference)
    return float(reference - current)


metric_direction_assertions = pd.DataFrame([
    {"case": "accuracy rises", "metric": "macro_f1", "delta": signed_improvement(0.90, 0.80, METRIC_SPECS["macro_f1"]), "expected": "positive"},
    {"case": "ECE falls", "metric": "ece", "delta": signed_improvement(0.05, 0.12, METRIC_SPECS["ece"]), "expected": "positive"},
    {"case": "latency rises", "metric": "latency_ms", "delta": signed_improvement(18.0, 12.0, METRIC_SPECS["latency_ms"]), "expected": "negative"},
])
assert metric_direction_assertions.loc[0, "delta"] > 0
assert metric_direction_assertions.loc[1, "delta"] > 0
assert metric_direction_assertions.loc[2, "delta"] < 0

SHIFTS = (
    ShiftContract("camera_B", "A", "B", "mild sensor/color mixing", "new approved synonyms", "none", "development_only"),
    ShiftContract("camera_C", "A", "C", "stronger held-out mixing", "same frozen vocabulary", "none", "reporting_only_no_changes"),
)
assert SHIFTS[1].evaluation_role == "reporting_only_no_changes"
metric_direction_assertions


## 3. Synthetic multimodal source shift

Each label has a latent visual prototype and a one-hot text concept. Site transforms alter visual observations but not the intended class relation. This is a controlled approximation, not proof of pure covariate shift. Site C is created now but never enters method, rank, threshold, or replay selection.

![Sensor, environment, object, task, language, policy, and embodiment shifts require different diagnoses.](assets/shift-taxonomy.svg)


In [ ]:
CLASS_NAMES = ("scratch", "dent", "seal_gap")
INPUT_DIM = 8
EMBED_DIM = 6

BASE_CENTERS = torch.tensor([
    [2.4, 0.1, 0.0, 0.8, -0.2, 0.3, 0.0, 0.4],
    [0.0, 2.3, 0.2, -0.4, 0.9, 0.0, 0.3, -0.2],
    [0.2, 0.0, 2.5, 0.1, -0.3, 0.9, -0.4, 0.0],
], dtype=torch.float32)

SITE_TRANSFORMS = {
    "A": torch.eye(INPUT_DIM),
    "B": torch.tensor([
        [0.62, 0.28, 0.10, 0, 0, 0, 0, 0],
        [0.12, 0.64, 0.24, 0, 0, 0, 0, 0],
        [0.26, 0.08, 0.66, 0, 0, 0, 0, 0],
        [0, 0, 0, 1, .18, 0, 0, 0],
        [0, 0, 0, -.12, 1, 0, 0, 0],
        [0, 0, 0, 0, 0, 1, .15, 0],
        [0, 0, 0, 0, 0, -.10, 1, 0],
        [0, 0, 0, 0, 0, 0, 0, 1],
    ], dtype=torch.float32),
    "C": torch.tensor([
        [0.45, 0.38, 0.17, 0, 0, 0, 0, 0],
        [0.24, 0.48, 0.28, 0, 0, 0, 0, 0],
        [0.34, 0.18, 0.48, 0, 0, 0, 0, 0],
        [0, 0, 0, .88, .30, 0, 0, 0],
        [0, 0, 0, -.22, .92, 0, 0, 0],
        [0, 0, 0, 0, 0, .90, .28, 0],
        [0, 0, 0, 0, 0, -.20, .92, 0],
        [0, 0, 0, 0, 0, 0, 0, 1],
    ], dtype=torch.float32),
}
SITE_OFFSETS = {
    "A": torch.zeros(INPUT_DIM),
    "B": torch.tensor([0.15, -0.10, 0.12, 0.30, -0.20, 0.10, 0.0, 0.15]),
    "C": torch.tensor([0.24, -0.18, 0.20, 0.42, -0.32, 0.20, -0.10, 0.24]),
}


@dataclass(frozen=True)
class SiteData:
    site: str
    role: str
    x: torch.Tensor
    y: torch.Tensor


def make_site(site: str, role: str, n_per_class: int, seed: int) -> SiteData:
    generator = torch.Generator().manual_seed(seed)
    rows, labels = [], []
    for label, center in enumerate(BASE_CENTERS):
        noise = torch.randn((n_per_class, INPUT_DIM), generator=generator) * 0.30
        raw = center.repeat(n_per_class, 1) + noise
        shifted = raw @ SITE_TRANSFORMS[site].T + SITE_OFFSETS[site]
        rows.append(shifted)
        labels.extend([label] * n_per_class)
    x = torch.cat(rows)
    y = torch.tensor(labels, dtype=torch.long)
    order = torch.randperm(len(y), generator=generator)
    return SiteData(site, role, x[order], y[order])


site_a_train = make_site("A", "legacy_train", 42, SEED + 1)
site_a_eval = make_site("A", "legacy", 28, SEED + 2)
site_b_train = make_site("B", "development_only", 24, SEED + 3)
site_b_eval = make_site("B", "development_only", 28, SEED + 4)
site_c_eval = make_site("C", "reporting_only_no_changes", 34, SEED + 5)

split_manifest = pd.DataFrame([
    {"site": data.site, "role": data.role, "rows": len(data.y), "used_for_training": data is site_a_train or data is site_b_train}
    for data in (site_a_train, site_a_eval, site_b_train, site_b_eval, site_c_eval)
])
assert split_manifest.query("site == 'C'")["used_for_training"].eq(False).all()
split_manifest


## 4. Train and freeze the validated base contract

The tiny dual encoder learns image and text embeddings with a contrastive class objective. The base is trained only on Site A. We immediately hash its weights and save immutable baseline metrics across all three sites; Site B/C rows are diagnostic at this stage, not adaptation evidence.


In [ ]:
class TinyDualEncoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.vision = nn.Linear(INPUT_DIM, EMBED_DIM)
        self.text = nn.Linear(len(CLASS_NAMES), EMBED_DIM, bias=False)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(8.0)))

    def image_embedding(self, x: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.vision(x), dim=-1)

    def text_embeddings(self) -> torch.Tensor:
        concepts = torch.eye(len(CLASS_NAMES), device=self.text.weight.device)
        return F.normalize(self.text(concepts), dim=-1)

    def class_logits(self, x: torch.Tensor) -> torch.Tensor:
        return self.logit_scale.exp().clamp(max=50) * self.image_embedding(x) @ self.text_embeddings().T


def state_hash(model: nn.Module) -> str:
    digest = hashlib.sha256()
    for name, value in sorted(model.state_dict().items()):
        digest.update(name.encode())
        digest.update(value.detach().cpu().numpy().tobytes())
    return digest.hexdigest()


def train_supervised(model: nn.Module, data: SiteData, epochs: int = 120, lr: float = 0.03, penalty=None) -> list[float]:
    parameters = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(parameters, lr=lr)
    history = []
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = F.cross_entropy(model.class_logits(data.x), data.y)
        if penalty is not None:
            loss = loss + penalty(model)
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach()))
    return history


base_model = TinyDualEncoder().to(DEVICE)
base_history = train_supervised(base_model, site_a_train, epochs=150, lr=0.025)
BASE_WEIGHTS_HASH = state_hash(base_model)
BASE_CONTRACT = BaseModelContract(
    model_id="local_tiny_dual_encoder",
    revision="advanced06-notebook-v1",
    processor_revision="synthetic-features-v1",
    weights_hash=BASE_WEIGHTS_HASH,
    capability_suite_version=CAPABILITY_SUITE.version,
    training_scope_known=True,
    license="MIT repository teaching code; generated synthetic data",
)
assert base_history[-1] < base_history[0]
BASE_CONTRACT


In [ ]:
def expected_calibration_error(probabilities: np.ndarray, labels: np.ndarray, bins: int = 8) -> float:
    confidence = probabilities.max(axis=1)
    predicted = probabilities.argmax(axis=1)
    correct = predicted == labels
    edges = np.linspace(0.0, 1.0, bins + 1)
    ece = 0.0
    for low, high in zip(edges[:-1], edges[1:]):
        mask = (confidence > low) & (confidence <= high)
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
    return float(ece)


@torch.no_grad()
def evaluate_capabilities(model: nn.Module, data: SiteData) -> dict[str, float | str]:
    logits = model.class_logits(data.x)
    probabilities = logits.softmax(dim=1).cpu().numpy()
    predictions = probabilities.argmax(axis=1)
    labels = data.y.cpu().numpy()
    image_z = model.image_embedding(data.x)
    text_z = model.text_embeddings()
    similarities = image_z @ text_z.T
    image_to_text = similarities.argmax(dim=1)
    text_to_image_correct = []
    for class_id in range(len(CLASS_NAMES)):
        top_image = int(similarities[:, class_id].argmax())
        text_to_image_correct.append(int(data.y[top_image]) == class_id)
    paired = similarities[torch.arange(len(data.y)), data.y]
    return {
        "site": data.site,
        "role": data.role,
        "accuracy": float(accuracy_score(labels, predictions)),
        "macro_f1": float(f1_score(labels, predictions, average="macro")),
        "image_to_text_accuracy": float((image_to_text == data.y).float().mean()),
        "text_to_image_accuracy": float(np.mean(text_to_image_correct)),
        "paired_cosine": float(paired.mean()),
        "ece": expected_calibration_error(probabilities, labels),
    }


base_metrics = pd.DataFrame([
    evaluate_capabilities(base_model, site_a_eval),
    evaluate_capabilities(base_model, site_b_eval),
    evaluate_capabilities(base_model, site_c_eval),
])
immutable_baseline_hash = canonical_hash(base_metrics.to_dict(orient="records"))
assert state_hash(base_model) == BASE_WEIGHTS_HASH
base_metrics


## 5. Adaptation surfaces: probe, projector, adapter, prompt, LoRA, partial, full

All methods start from the exact base digest and see the same Site B rows, epochs, and evaluation suite. `AdaptedDualEncoder` makes the trainable surface explicit. Parameter efficiency and capability preservation are measured rather than assumed.

![Adaptation surfaces progress from a frozen representation to modular capacity and broader overwrite.](assets/adaptation-methods.svg)


In [ ]:
def lora_parameter_count(d: int, k: int, rank: int) -> int:
    if rank <= 0 or rank > min(d, k):
        raise ValueError("rank must be in [1, min(d, k)]")
    return rank * (d + k)


lora_counts = pd.DataFrame([
    {"rank": rank, "full_matrix_parameters": 64 * 48, "lora_parameters": lora_parameter_count(64, 48, rank)}
    for rank in (1, 2, 4, 8, 16)
])
assert lora_parameter_count(8, 6, 2) == 28
lora_counts


In [ ]:
class AdaptedDualEncoder(nn.Module):
    def __init__(self, base: TinyDualEncoder, method: str, rank: int = 2, adapter_width: int = 3) -> None:
        super().__init__()
        self.method = method
        self.base = copy.deepcopy(base)
        for parameter in self.base.parameters():
            parameter.requires_grad = False
        self.head = None
        self.adapter_down = None
        self.adapter_up = None
        self.input_prompt = None
        self.lora_A = None
        self.lora_B = None
        self.lora_scale = 1.0 / rank

        if method == "linear_probe":
            self.head = nn.Linear(EMBED_DIM, len(CLASS_NAMES))
        elif method == "projector":
            for parameter in self.base.vision.parameters():
                parameter.requires_grad = True
        elif method == "adapter":
            self.adapter_down = nn.Linear(EMBED_DIM, adapter_width, bias=False)
            self.adapter_up = nn.Linear(adapter_width, EMBED_DIM, bias=False)
            nn.init.zeros_(self.adapter_up.weight)
        elif method == "visual_prompt":
            self.input_prompt = nn.Parameter(torch.zeros(INPUT_DIM))
        elif method == "lora":
            self.lora_A = nn.Parameter(torch.empty(rank, INPUT_DIM))
            self.lora_B = nn.Parameter(torch.zeros(EMBED_DIM, rank))
            nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        elif method == "partial_ft":
            for parameter in self.base.vision.parameters():
                parameter.requires_grad = True
            self.base.logit_scale.requires_grad = True
        elif method == "full_ft":
            for parameter in self.base.parameters():
                parameter.requires_grad = True
        else:
            raise ValueError(f"unknown method: {method}")

    def image_embedding(self, x: torch.Tensor) -> torch.Tensor:
        if self.input_prompt is not None:
            x = x + self.input_prompt
        if self.lora_A is not None and self.lora_B is not None:
            update = self.lora_B @ self.lora_A
            hidden = F.linear(x, self.base.vision.weight + self.lora_scale * update, self.base.vision.bias)
        else:
            hidden = self.base.vision(x)
        if self.adapter_down is not None and self.adapter_up is not None:
            hidden = hidden + self.adapter_up(F.gelu(self.adapter_down(hidden)))
        return F.normalize(hidden, dim=-1)

    def text_embeddings(self) -> torch.Tensor:
        return self.base.text_embeddings()

    def class_logits(self, x: torch.Tensor) -> torch.Tensor:
        image_z = self.image_embedding(x)
        if self.head is not None:
            return self.head(image_z)
        return self.base.logit_scale.exp().clamp(max=50) * image_z @ self.text_embeddings().T


def trainable_parameter_names(model: nn.Module) -> tuple[str, ...]:
    return tuple(name for name, parameter in model.named_parameters() if parameter.requires_grad)


def train_candidate(method: str, rank: int = 2, epochs: int = 90) -> tuple[AdaptedDualEncoder, dict[str, float]]:
    torch.manual_seed(SEED + len(method) + rank)
    candidate = AdaptedDualEncoder(base_model, method, rank=rank)
    start = time.perf_counter()
    history = train_supervised(candidate, site_b_train, epochs=epochs, lr=0.025)
    elapsed = time.perf_counter() - start
    trainable = sum(p.numel() for p in candidate.parameters() if p.requires_grad)
    return candidate, {
        "final_training_loss": history[-1],
        "trainable_parameters": trainable,
        "artifact_bytes_fp32": trainable * 4,
        "train_time_s": elapsed,
    }


# A zero-initialized LoRA B matrix is an exact no-op before training.
lora_noop = AdaptedDualEncoder(base_model, "lora", rank=2)
with torch.no_grad():
    assert torch.allclose(lora_noop.image_embedding(site_a_eval.x), base_model.image_embedding(site_a_eval.x), atol=1e-7)


## 6. Matched method comparison and rank capacity

The rank sweep uses Site B only. Site C remains unopened for selection. A higher rank means more update capacity, not guaranteed target gain or retention.


In [ ]:
rank_sweep_rows = []
for rank in (1, 2, 4):
    candidate, systems = train_candidate("lora", rank=rank, epochs=70)
    metrics = evaluate_capabilities(candidate, site_b_eval)
    rank_sweep_rows.append({"rank": rank, **systems, **metrics})
rank_sweep = pd.DataFrame(rank_sweep_rows)
SELECTED_LORA_RANK = int(rank_sweep.sort_values(["macro_f1", "trainable_parameters"], ascending=[False, True]).iloc[0]["rank"])
rank_sweep


In [ ]:
METHODS = ("linear_probe", "projector", "adapter", "visual_prompt", "lora", "partial_ft", "full_ft")
candidates: dict[str, AdaptedDualEncoder] = {}
comparison_rows = []
for method in METHODS:
    candidate, systems = train_candidate(method, rank=SELECTED_LORA_RANK, epochs=90)
    candidates[method] = candidate
    for data in (site_a_eval, site_b_eval):
        comparison_rows.append({"method": method, **systems, **evaluate_capabilities(candidate, data)})

method_comparison = pd.DataFrame(comparison_rows)
base_by_site = base_metrics.set_index("site")
method_comparison["transfer_or_regression_accuracy"] = method_comparison.apply(
    lambda row: row["accuracy"] - float(base_by_site.loc[row["site"], "accuracy"]), axis=1
)
method_comparison["alignment_change"] = method_comparison.apply(
    lambda row: row["image_to_text_accuracy"] - float(base_by_site.loc[row["site"], "image_to_text_accuracy"]), axis=1
)
method_comparison.sort_values(["site", "macro_f1"], ascending=[True, False])


Interpret Site B differences as target transfer and Site A differences as legacy regression. Trainable-parameter count is a systems property; it does not replace behavioral evidence.


In [ ]:
method_summary = method_comparison.pivot(index="method", columns="site", values=[
    "accuracy", "macro_f1", "image_to_text_accuracy", "text_to_image_accuracy", "paired_cosine", "ece"
])
method_summary.columns = [f"{metric}_{site}" for metric, site in method_summary.columns]
method_summary = method_summary.reset_index()
method_summary["target_gain_B"] = method_summary["macro_f1_B"] - float(base_by_site.loc["B", "macro_f1"])
method_summary["legacy_regression_A"] = method_summary["macro_f1_A"] - float(base_by_site.loc["A", "macro_f1"])
method_summary["legacy_alignment_regression_A"] = method_summary["image_to_text_accuracy_A"] - float(base_by_site.loc["A", "image_to_text_accuracy"])

systems_once = method_comparison.groupby("method", as_index=False).first()[["method", "trainable_parameters", "artifact_bytes_fp32", "train_time_s", "final_training_loss"]]
method_summary = method_summary.merge(systems_once, on="method")
method_summary.sort_values("target_gain_B", ascending=False)


## 7. Representation, neighborhood, and alignment drift

Cosine movement alone cannot say whether change was useful. We pair it with nearest-neighbor retention and capability metrics on the same fixed legacy examples. Three controls make the stability–plasticity lesson explicit: a frozen model with almost no movement and no adaptation gain, a bounded adapter, and unconstrained full fine-tuning. The desired candidate is not the one with minimum drift; it moves enough to gain the target capability while preserving validated legacy structure.

![A moved image tower can improve one task while regressing cross-modal geometry.](assets/multimodal-alignment-drift.svg)


In [ ]:
@torch.no_grad()
def neighborhood_retention(before: torch.Tensor, after: torch.Tensor, k: int = 5) -> float:
    before_sim = before @ before.T
    after_sim = after @ after.T
    before_sim.fill_diagonal_(-torch.inf)
    after_sim.fill_diagonal_(-torch.inf)
    before_neighbors = before_sim.topk(k, dim=1).indices
    after_neighbors = after_sim.topk(k, dim=1).indices
    overlaps = []
    for row in range(len(before)):
        overlaps.append(len(set(before_neighbors[row].tolist()) & set(after_neighbors[row].tolist())) / k)
    return float(np.mean(overlaps))


@torch.no_grad()
def drift_report(method: str, candidate: nn.Module) -> dict[str, float | str]:
    before = base_model.image_embedding(site_a_eval.x)
    after = candidate.image_embedding(site_a_eval.x)
    cosine = F.cosine_similarity(before, after)
    legacy = evaluate_capabilities(candidate, site_a_eval)
    target = evaluate_capabilities(candidate, site_b_eval)
    return {
        "method": method,
        "mean_embedding_cosine": float(cosine.mean()),
        "mean_cosine_drift": float((1 - cosine).mean()),
        "nearest_neighbor_retention_at_5": neighborhood_retention(before, after, k=5),
        "legacy_image_to_text_accuracy": legacy["image_to_text_accuracy"],
        "legacy_text_to_image_accuracy": legacy["text_to_image_accuracy"],
        "legacy_paired_cosine": legacy["paired_cosine"],
        "target_B_macro_f1": target["macro_f1"],
        "target_B_gain": target["macro_f1"] - float(base_by_site.loc["B", "macro_f1"]),
    }


representation_drift = pd.DataFrame([drift_report(name, model) for name, model in candidates.items()])
stability_plasticity_controls = pd.DataFrame([
    {**drift_report("A_frozen_base", base_model), "interpretation": "minimum movement, but no adaptation gain"},
    {**drift_report("B_bounded_adapter", candidates["adapter"]), "interpretation": "bounded plasticity; inspect gain and retained structure together"},
    {**drift_report("C_broad_projector_tuning", candidates["projector"]), "interpretation": "more movement and target gain, but weaker legacy neighborhoods"},
])
assert stability_plasticity_controls.loc[stability_plasticity_controls["mean_cosine_drift"].idxmin(), "method"] == "A_frozen_base"
assert stability_plasticity_controls.loc[stability_plasticity_controls["target_B_macro_f1"].idxmax(), "method"] != "A_frozen_base"
assert stability_plasticity_controls.loc[2, "target_B_gain"] > 0
assert stability_plasticity_controls.loc[2, "nearest_neighbor_retention_at_5"] < stability_plasticity_controls.loc[1, "nearest_neighbor_retention_at_5"]
representation_drift.sort_values("mean_cosine_drift", ascending=False), stability_plasticity_controls


## 8. Low adaptation loss is not a capability contract

Compare candidates with their training loss and capability suite. A candidate may optimize Site B examples while changing legacy retrieval geometry. Loss is an optimization signal, not a release receipt.


In [ ]:
pretext_vs_downstream = method_summary[[
    "method", "final_training_loss", "macro_f1_B", "target_gain_B",
    "macro_f1_A", "legacy_regression_A", "image_to_text_accuracy_A", "legacy_alignment_regression_A"
]].sort_values("final_training_loss")

# Failure injection: the shortcut-only adaptation set encodes labels in a
# permuted channel pattern that is absent from normal evaluation. Its objective
# can look excellent while legacy and target capability degrade.
shortcut_x = torch.zeros_like(site_b_train.x)
permuted_shortcut = F.one_hot((site_b_train.y + 1) % len(CLASS_NAMES), num_classes=len(CLASS_NAMES)).float()
shortcut_x[:, : len(CLASS_NAMES)] = 6.0 * permuted_shortcut
shortcut_train = SiteData("B-shortcut", "failure_injection", shortcut_x, site_b_train.y)
shortcut_candidate = AdaptedDualEncoder(base_model, "full_ft")
shortcut_history = train_supervised(shortcut_candidate, shortcut_train, epochs=140, lr=0.035)

reference_method = "adapter"
reference_loss = float(method_summary.set_index("method").loc[reference_method, "final_training_loss"])
low_loss_candidates = []
for name, model, objective_loss in (
    ("normal_site_B_adapter", candidates[reference_method], reference_loss),
    ("shortcut_only_full_ft", shortcut_candidate, shortcut_history[-1]),
):
    a_metrics = evaluate_capabilities(model, site_a_eval)
    b_metrics = evaluate_capabilities(model, site_b_eval)
    low_loss_candidates.append({
        "candidate": name,
        "adaptation_objective_loss": objective_loss,
        "legacy_A_macro_f1": a_metrics["macro_f1"],
        "legacy_A_image_to_text": a_metrics["image_to_text_accuracy"],
        "target_B_macro_f1": b_metrics["macro_f1"],
        "target_B_image_to_text": b_metrics["image_to_text_accuracy"],
    })
low_loss_candidates = pd.DataFrame(low_loss_candidates)
assert low_loss_candidates.loc[1, "adaptation_objective_loss"] < 0.02
pretext_vs_downstream, low_loss_candidates


## 9. Select on Site B, then freeze before Site C

The development gate first requires bounded Site A regression, then ranks eligible methods by Site B macro F1 and trainable footprint. This is demonstration policy, not a universal threshold. Its canonical hash is recorded before Site C evaluation.


In [ ]:
DEMONSTRATION_THRESHOLD_NOTICE = "Demonstration thresholds for this synthetic notebook only."
development_policy = {
    "version": "advanced06-development-policy-v1",
    "selection_source": "Site B development only",
    "legacy_macro_f1_regression_min": -0.20,
    "legacy_alignment_regression_min": -0.20,
    "rank_tiebreak": "fewer trainable parameters",
    "site_c_role": "reporting only; no method, rank, prompt, threshold, buffer, epoch, seed, or rule changes",
    "notice": DEMONSTRATION_THRESHOLD_NOTICE,
}
eligible = method_summary[
    (method_summary["legacy_regression_A"] >= development_policy["legacy_macro_f1_regression_min"])
    & (method_summary["legacy_alignment_regression_A"] >= development_policy["legacy_alignment_regression_min"])
].copy()
if eligible.empty:
    raise AssertionError("No candidate satisfied the development retention policy")

selected_row = eligible.sort_values(["macro_f1_B", "trainable_parameters"], ascending=[False, True]).iloc[0]
SELECTED_METHOD = str(selected_row["method"])
selected_candidate = candidates[SELECTED_METHOD]
FROZEN_POLICY_HASH = canonical_hash(development_policy)
policy_hash_before_site_c = FROZEN_POLICY_HASH

site_c_candidate_metrics = evaluate_capabilities(selected_candidate, site_c_eval)
policy_hash_after_site_c = canonical_hash(development_policy)
assert policy_hash_before_site_c == policy_hash_after_site_c
assert site_c_eval.role == "reporting_only_no_changes"

site_c_report = pd.DataFrame([
    {"model": "base", **evaluate_capabilities(base_model, site_c_eval)},
    {"model": f"candidate:{SELECTED_METHOD}", **site_c_candidate_metrics},
])
site_c_report


## 10. Continual learning: A → B → C

We now move from one adaptation event to a separate controlled sequence. Its third experience is named `CL-C`; it is **not** held-out Site C. Sequential full fine-tuning is the baseline. Replay, EWC-style regularization, and distillation use different retention mechanisms. Metrics are recorded after every experience, producing the central forgetting matrix while the reporting-only Site C remains untouched.

Every continual statistic carries its metric name, direction, numeric reference, and reference semantics. `signed_improvement` makes positive mean better for both higher-is-better macro F1 and lower-is-better ECE. Forgetting is the non-negative regression magnitude from the best **previous** task-A checkpoint; BWT uses task-A performance immediately after learning A; the FWT proxy names its pre-CL-C reference explicitly. A proxy is not presented as canonical FWT without a single-task control.

![The continual learner must balance acquisition and retention.](assets/stability-plasticity.svg)


In [ ]:
def concatenate_sites(name: str, role: str, items: list[SiteData]) -> SiteData:
    return SiteData(name, role, torch.cat([item.x for item in items]), torch.cat([item.y for item in items]))


def make_continual_c(n_per_class: int, seed: int, role: str) -> SiteData:
    base = make_site("A", role, n_per_class, seed)
    mixed = base.x.clone()
    # A strong domain-incremental feature rotation: labels stay fixed, but the
    # dominant class-bearing channels move. It is intentionally hard enough to
    # expose stability/plasticity trade-offs in a tiny linear teaching model.
    mixed[:, :3] = 0.08 * base.x[:, :3] + 0.92 * base.x[:, [1, 2, 0]]
    mixed[:, 3:] = base.x[:, 3:] + torch.tensor([0.35, -0.25, 0.18, -0.12, 0.22])
    return SiteData("CL-C", role, mixed, base.y)


continual_c_train = make_continual_c(24, SEED + 40, "continual_training_experience")
continual_c_eval = make_continual_c(28, SEED + 41, "continual_evaluation")


def source_class_balanced_replay(
    items: list[SiteData], capacity: int, seed: int
) -> tuple[SiteData | None, pd.DataFrame]:
    if capacity <= 0:
        return None, pd.DataFrame(columns=["source", "class_id", "selected"])
    generator = torch.Generator().manual_seed(seed)
    # Interleave sources so any remainder is distributed across sources first.
    strata = [(item, class_id) for class_id in range(len(CLASS_NAMES)) for item in items]
    base_quota, remainder = divmod(capacity, len(strata))
    selected_x, selected_y, audit_rows = [], [], []
    for position, (item, class_id) in enumerate(strata):
        quota = base_quota + int(position < remainder)
        choices = torch.where(item.y == class_id)[0]
        order = choices[torch.randperm(len(choices), generator=generator)]
        chosen = order[: min(quota, len(order))]
        selected_x.append(item.x[chosen])
        selected_y.append(item.y[chosen])
        audit_rows.append({"source": item.site, "class_id": class_id, "selected": len(chosen)})
    if not selected_x:
        return None, pd.DataFrame(audit_rows)
    replay = SiteData("replay", "governed_replay", torch.cat(selected_x), torch.cat(selected_y))
    return replay, pd.DataFrame(audit_rows)


def balanced_replay(items: list[SiteData], capacity: int, seed: int) -> SiteData | None:
    replay, _ = source_class_balanced_replay(items, capacity, seed)
    return replay


def diagonal_fisher(model: TinyDualEncoder, data: SiteData) -> dict[str, torch.Tensor]:
    model.zero_grad()
    loss = F.cross_entropy(model.class_logits(data.x), data.y)
    loss.backward()
    return {name: parameter.grad.detach().pow(2).clone() if parameter.grad is not None else torch.zeros_like(parameter) for name, parameter in model.named_parameters()}


def train_continual_step(
    model: TinyDualEncoder,
    current: SiteData,
    replay: SiteData | None = None,
    fisher: dict[str, torch.Tensor] | None = None,
    anchor: dict[str, torch.Tensor] | None = None,
    ewc_lambda: float = 0.0,
    teacher: TinyDualEncoder | None = None,
    distill_weight: float = 0.0,
    epochs: int = 85,
) -> None:
    train_data = current if replay is None else concatenate_sites("current_plus_replay", "training", [current, replay])
    optimizer = torch.optim.Adam(model.parameters(), lr=0.018)
    if teacher is not None:
        teacher.eval()
    for _ in range(epochs):
        optimizer.zero_grad()
        logits = model.class_logits(train_data.x)
        loss = F.cross_entropy(logits, train_data.y)
        if fisher is not None and anchor is not None:
            penalty = sum((fisher[name] * (parameter - anchor[name]).pow(2)).sum() for name, parameter in model.named_parameters())
            loss = loss + 0.5 * ewc_lambda * penalty
        if teacher is not None and distill_weight > 0:
            with torch.no_grad():
                teacher_probs = teacher.class_logits(current.x).softmax(dim=1)
            student_log_probs = model.class_logits(current.x).log_softmax(dim=1)
            loss = loss + distill_weight * F.kl_div(student_log_probs, teacher_probs, reduction="batchmean")
        loss.backward()
        optimizer.step()


def continual_run(strategy: str, replay_capacity: int = 0) -> tuple[pd.DataFrame, TinyDualEncoder]:
    model = copy.deepcopy(base_model)
    evaluation_sites = (site_a_eval, site_b_eval, continual_c_eval)
    records = []

    def record(after_experience: str) -> None:
        for data in evaluation_sites:
            result = evaluate_capabilities(model, data)
            records.append({"strategy": strategy, "after_experience": after_experience, **result})

    record("A")
    seen = [site_a_train]
    fisher = diagonal_fisher(model, site_a_train) if strategy == "ewc" else None
    anchor = {name: value.detach().clone() for name, value in model.named_parameters()} if strategy == "ewc" else None

    for step, current in enumerate((site_b_train, continual_c_train), start=1):
        replay = balanced_replay(seen, replay_capacity, SEED + 100 + step) if strategy.startswith("replay") else None
        teacher = copy.deepcopy(model) if strategy == "distillation" else None
        train_continual_step(
            model,
            current,
            replay=replay,
            fisher=fisher,
            anchor=anchor,
            ewc_lambda=12.0 if strategy == "ewc" else 0.0,
            teacher=teacher,
            distill_weight=1.2 if strategy == "distillation" else 0.0,
        )
        seen.append(current)
        if strategy == "ewc":
            new_fisher = diagonal_fisher(model, current)
            fisher = {name: fisher[name] + new_fisher[name] for name in fisher}
            anchor = {name: value.detach().clone() for name, value in model.named_parameters()}
        record("B" if step == 1 else "C")
    return pd.DataFrame(records), model


sequential_ft_matrix, sequential_ft_model = continual_run("sequential_full_finetune")
replay_50_matrix, replay_50_model = continual_run("replay_50", replay_capacity=50)
ewc_matrix, ewc_model = continual_run("ewc")
distillation_matrix, distillation_model = continual_run("distillation")
continual_results = pd.concat([sequential_ft_matrix, replay_50_matrix, ewc_matrix, distillation_matrix], ignore_index=True)
continual_results.head()


In [ ]:
def metric_matrix(frame: pd.DataFrame, strategy: str, metric: str = "macro_f1") -> pd.DataFrame:
    subset = frame.query("strategy == @strategy")
    return subset.pivot(index="site", columns="after_experience", values=metric).reindex(index=["A", "B", "CL-C"], columns=["A", "B", "C"])


forgetting_matrix = metric_matrix(continual_results, "replay_50")

def best_value(values: pd.Series, metric_spec: MetricSpec) -> float:
    return float(values.max() if metric_spec.direction == "higher_is_better" else values.min())


def continual_metric_records(
    frame: pd.DataFrame,
    strategy: str,
    metric_spec: MetricSpec,
) -> list[dict[str, float | str]]:
    '''Store values and exact references; positive signed deltas always mean improvement.'''
    matrix = metric_matrix(frame, strategy, metric_spec.name)
    best_previous_a = best_value(matrix.loc["A", ["A", "B"]], metric_spec)
    current_a = float(matrix.loc["A", "C"])
    bwt_reference_a = float(matrix.loc["A", "A"])
    fwt_reference_cl_c = float(matrix.loc["CL-C", "A"])
    fwt_current_cl_c = float(matrix.loc["CL-C", "B"])
    forgetting_signed_improvement = signed_improvement(current_a, best_previous_a, metric_spec)
    return [
        {
            "strategy": strategy,
            "statistic": "forgetting_A",
            "metric": metric_spec.name,
            "direction": metric_spec.direction,
            "reference_semantics": "best performance on task A before the current checkpoint (after A or B)",
            "reference": best_previous_a,
            "current": current_a,
            "signed_improvement": forgetting_signed_improvement,
            "value": max(0.0, -forgetting_signed_improvement),
            "value_semantics": "non-negative regression magnitude; larger is worse",
        },
        {
            "strategy": strategy,
            "statistic": "backward_transfer_A",
            "metric": metric_spec.name,
            "direction": metric_spec.direction,
            "reference_semantics": "task A performance immediately after learning task A",
            "reference": bwt_reference_a,
            "current": current_a,
            "signed_improvement": signed_improvement(current_a, bwt_reference_a, metric_spec),
            "value": signed_improvement(current_a, bwt_reference_a, metric_spec),
            "value_semantics": "positive is beneficial backward transfer; negative is regression",
        },
        {
            "strategy": strategy,
            "statistic": "forward_transfer_CL_C_proxy",
            "metric": metric_spec.name,
            "direction": metric_spec.direction,
            "reference_semantics": "CL-C performance after A, before learning B; proxy reference, not a single-task control",
            "reference": fwt_reference_cl_c,
            "current": fwt_current_cl_c,
            "signed_improvement": signed_improvement(fwt_current_cl_c, fwt_reference_cl_c, metric_spec),
            "value": signed_improvement(fwt_current_cl_c, fwt_reference_cl_c, metric_spec),
            "value_semantics": "positive means learning B helped CL-C before CL-C training",
        },
    ]


continual_metric_evidence = pd.DataFrame([
    record
    for strategy in continual_results["strategy"].unique()
    for metric_spec in (METRIC_SPECS["macro_f1"], METRIC_SPECS["ece"])
    for record in continual_metric_records(continual_results, strategy, metric_spec)
])
assert set(continual_metric_evidence["direction"]) == {"higher_is_better", "lower_is_better"}
assert continual_metric_evidence["reference_semantics"].str.len().gt(0).all()

macro_records = continual_metric_evidence.query("metric == 'macro_f1'")
continual_summary = macro_records.pivot(index="strategy", columns="statistic", values="value").reset_index()
continual_summary["final_mean_macro_f1"] = continual_summary["strategy"].map({
    strategy: float(metric_matrix(continual_results, strategy)["C"].mean())
    for strategy in continual_results["strategy"].unique()
})
continual_summary = continual_summary.rename(columns={"forward_transfer_CL_C_proxy": "forward_transfer_C_proxy"})
forgetting_matrix, continual_summary, continual_metric_evidence


## 11. Replay-buffer size sweep

Replay is a governed data strategy. The sweep reports new-task quality, legacy retention, retained-example count, and measured training time. The buffer selector balances both source and class and emits a membership audit, because a class-balanced buffer can still omit a legacy source. Random, class-balanced, and source-class-balanced replay make different representativeness assumptions. More retained examples can help, but privacy, licensing, deletion, and source balance remain release constraints.


In [ ]:
replay_sweep_rows = []
for capacity in (0, 10, 50, 200):
    start = time.perf_counter()
    matrix_frame, _ = continual_run("replay", replay_capacity=capacity)
    elapsed = time.perf_counter() - start
    matrix = metric_matrix(matrix_frame, "replay")
    replay_sweep_rows.append({
        "replay_capacity": capacity,
        "legacy_A_final_macro_f1": float(matrix.loc["A", "C"]),
        "new_C_final_macro_f1": float(matrix.loc["CL-C", "C"]),
        "forgetting_A": max(0.0, -signed_improvement(
            float(matrix.loc["A", "C"]),
            best_value(matrix.loc["A", ["A", "B"]], METRIC_SPECS["macro_f1"]),
            METRIC_SPECS["macro_f1"],
        )),
        "retained_examples": min(capacity, len(site_a_train.y) + len(site_b_train.y)),
        "measured_train_time_s": elapsed,
    })
replay_size_sweep = pd.DataFrame(replay_sweep_rows)

buffer_example, replay_selection_audit = source_class_balanced_replay([site_a_train, site_b_train], 50, SEED + 500)
assert buffer_example is not None
assert replay_selection_audit.groupby("source")["selected"].sum().max() - replay_selection_audit.groupby("source")["selected"].sum().min() <= 1
REPLAY_BUFFER_CONTRACT = ReplayBufferContract(
    buffer_version="3",
    selection_method="class_source_balanced_teaching_proxy",
    capacity=50,
    source_domains=("A", "B"),
    content_policy="synthetic_non_personal; production requires rights, retention, deletion, and access review",
    digest=canonical_hash({"x": buffer_example.x.tolist(), "y": buffer_example.y.tolist()}),
)
replay_size_sweep, replay_selection_audit


## 12. Parameter isolation and trusted routing

Separate adapters avoid overwriting each other, but they assume a correct route. Here the site identifier is trusted synthetic application context. Model-generated text never chooses authority or deployment scope. The executable cases compare correct routing, deliberate misrouting, an unknown domain, and an ambiguous domain; unknown or ambiguous context must abstain.

![Replay, regularization, distillation, and isolated adapters use different retention mechanisms.](assets/continual-learning-strategies.svg)


In [ ]:
isolated_adapters = {
    "A": AdaptedDualEncoder(base_model, "adapter"),
    "B": train_candidate("adapter", epochs=70)[0],
}
continual_c_adapter = AdaptedDualEncoder(base_model, "adapter")
train_supervised(continual_c_adapter, continual_c_train, epochs=70, lr=0.025)
isolated_adapters["CL-C"] = continual_c_adapter

def route_adapter(trusted_site_context: str) -> AdaptedDualEncoder | None:
    if trusted_site_context not in isolated_adapters:
        return None
    return isolated_adapters[trusted_site_context]


routing_scenarios = (
    ("correct_A", "A", site_a_eval, "A"),
    ("correct_B", "B", site_b_eval, "B"),
    ("wrong_B_for_A", "B", site_a_eval, "A"),
    ("unknown_domain", "unknown_site", site_a_eval, None),
    ("ambiguous_domain", "A|B", site_a_eval, None),
)
isolated_routing_rows = []
for scenario, supplied_context, evaluation_data, expected_route in routing_scenarios:
    routed = route_adapter(supplied_context)
    metrics = evaluate_capabilities(routed, evaluation_data) if routed is not None else {}
    selected_route = supplied_context if routed is not None else None
    isolated_routing_rows.append({
        "scenario": scenario,
        "trusted_context": supplied_context,
        "evaluation_site": evaluation_data.site,
        "expected_route": expected_route,
        "selected_route": selected_route,
        "routing_correct": selected_route == expected_route,
        "outcome": "abstain" if routed is None else "routed",
        "capability_macro_f1": metrics.get("macro_f1", np.nan),
    })
isolated_routing = pd.DataFrame(isolated_routing_rows)
known_route_mask = isolated_routing["expected_route"].notna()
routing_accuracy = float(isolated_routing.loc[known_route_mask, "routing_correct"].mean())
unknown_domain_abstention = float(isolated_routing.loc[~known_route_mask, "outcome"].eq("abstain").mean())
assert route_adapter("unknown_site") is None
assert route_adapter("A|B") is None
assert unknown_domain_abstention == 1.0
isolated_routing, {"routing_accuracy": routing_accuracy, "unknown_domain_abstention": unknown_domain_abstention}


## 13. Build an immutable candidate artifact and block base mismatch

Adapters and prompts are meaningful only relative to an exact base. The loader verifies base digest, capability-suite version, and frozen policy hash before evaluation. A deliberately corrupted base reference is rejected.


In [ ]:
selected_training_manifest = {
    "training_source": "Site B development only",
    "training_rows": len(site_b_train.y),
    "seed": SEED,
    "method": SELECTED_METHOD,
    "selected_lora_rank": SELECTED_LORA_RANK,
    "site_c_used_for_selection": False,
}
CANDIDATE = CandidateArtifact(
    candidate_id="advanced06-candidate-001",
    base_weights_hash=BASE_CONTRACT.weights_hash,
    candidate_weights_hash=state_hash(selected_candidate),
    method=SELECTED_METHOD,
    trainable_parameter_names=trainable_parameter_names(selected_candidate),
    adapter_config={"method": SELECTED_METHOD, "lora_rank": SELECTED_LORA_RANK, "target": "vision projection in local proxy"},
    training_manifest_hash=canonical_hash(selected_training_manifest),
    replay_buffer_digest=None,
    capability_suite_version=CAPABILITY_SUITE.version,
    frozen_policy_hash=FROZEN_POLICY_HASH,
)

def validate_candidate_contract(candidate: CandidateArtifact, base: BaseModelContract) -> None:
    if candidate.base_weights_hash != base.weights_hash:
        raise ValueError("base_checkpoint_digest_mismatch")
    if candidate.capability_suite_version != base.capability_suite_version:
        raise ValueError("capability_suite_version_mismatch")
    if candidate.frozen_policy_hash != FROZEN_POLICY_HASH:
        raise ValueError("frozen_policy_hash_mismatch")


validate_candidate_contract(CANDIDATE, BASE_CONTRACT)
corrupt_candidate = CandidateArtifact(**{**asdict(CANDIDATE), "base_weights_hash": "0" * 64})
wrong_base_blocked = False
try:
    validate_candidate_contract(corrupt_candidate, BASE_CONTRACT)
except ValueError as error:
    wrong_base_blocked = str(error) == "base_checkpoint_digest_mismatch"
assert wrong_base_blocked


## 14. Independent capability promotion gate

The gate consumes measured evidence and exact lineage. It does not trust a training-job string such as “ready.” Demonstration tolerances are fixed on Site B, then applied unchanged to Site C. Checks have three states: `PASS`, `FAIL`, and `MISSING`. Missing evidence can never become success. A successful outcome is only `promote_to_shadow`; production authorization remains absent.

![Candidate lineage and regression evidence feed a trusted release control with rollback.](assets/capability-promotion-gate.svg)


In [ ]:
release_policy = {
    "version": "advanced06-release-policy-v1",
    "target_macro_f1_gain_min": -0.02,
    "legacy_A_macro_f1_regression_min": -0.20,
    "legacy_A_alignment_regression_min": -0.20,
    "max_ece_increase": 0.20,
    "require_rollback_target": True,
    "decision_scope": "shadow only",
    "notice": DEMONSTRATION_THRESHOLD_NOTICE,
}

base_a = evaluate_capabilities(base_model, site_a_eval)
candidate_a = evaluate_capabilities(selected_candidate, site_a_eval)
base_c = evaluate_capabilities(base_model, site_c_eval)
candidate_c = evaluate_capabilities(selected_candidate, site_c_eval)

def gate_status(condition: bool) -> Literal["PASS", "FAIL"]:
    return "PASS" if condition else "FAIL"


def metric_gate_check(
    evidence: dict[str, float | None],
    evidence_key: str,
    check_name: str,
    threshold: float,
    comparison: Literal["minimum", "maximum"],
    detail: str,
) -> GateCheck:
    value = evidence.get(evidence_key)
    if value is None or not math.isfinite(float(value)):
        return GateCheck(check_name, "MISSING", f"{detail}; required evidence `{evidence_key}` is missing")
    passed = float(value) >= threshold if comparison == "minimum" else float(value) <= threshold
    return GateCheck(check_name, gate_status(passed), f"{detail}; observed={float(value):.6f}, threshold={threshold:.6f}")


def evaluate_promotion_gate(
    candidate_id: str,
    capability_evidence: dict[str, float | None],
    *,
    lineage_valid: bool,
    actual_suite_version: str,
    expected_suite_version: str,
    rollback_target_hash: str | None,
    policy_frozen: bool,
) -> PromotionDecision:
    checks = (
        GateCheck("lineage_valid", gate_status(lineage_valid), "exact base, candidate, and policy digests verified"),
        GateCheck("suite_version_match", gate_status(actual_suite_version == expected_suite_version), f"actual={actual_suite_version}; expected={expected_suite_version}"),
        metric_gate_check(capability_evidence, "target_macro_f1_gain", "target_gain", release_policy["target_macro_f1_gain_min"], "minimum", "Site C reporting-only target macro F1 gain"),
        metric_gate_check(capability_evidence, "legacy_A_macro_f1_regression", "legacy_retention", release_policy["legacy_A_macro_f1_regression_min"], "minimum", "Site A macro F1 signed change"),
        metric_gate_check(capability_evidence, "legacy_A_alignment_regression", "alignment_retention", release_policy["legacy_A_alignment_regression_min"], "minimum", "Site A image-to-text signed change"),
        metric_gate_check(capability_evidence, "site_C_ece_increase", "calibration", release_policy["max_ece_increase"], "maximum", "Site C ECE increase"),
        GateCheck("rollback_ready", gate_status(bool(rollback_target_hash)), "immutable rollback digest must be present"),
        GateCheck("site_c_policy_frozen", gate_status(policy_frozen), "no reporting-source tuning"),
    )
    statuses = {check.status for check in checks}
    hard_control_failed = any(check.status == "FAIL" for check in checks if check.name in {"lineage_valid", "suite_version_match", "rollback_ready", "site_c_policy_frozen"})
    if hard_control_failed or "FAIL" in statuses:
        decision: Literal["promote_to_shadow", "needs_review", "reject", "rollback"] = "reject"
    elif "MISSING" in statuses:
        decision = "needs_review"
    else:
        decision = "promote_to_shadow"
    return PromotionDecision(candidate_id, decision, checks, rollback_target_hash, "none")


measured_capability_evidence = {
    "target_macro_f1_gain": candidate_c["macro_f1"] - base_c["macro_f1"],
    "legacy_A_macro_f1_regression": candidate_a["macro_f1"] - base_a["macro_f1"],
    "legacy_A_alignment_regression": candidate_a["image_to_text_accuracy"] - base_a["image_to_text_accuracy"],
    "site_C_ece_increase": candidate_c["ece"] - base_c["ece"],
}
promotion_decision = evaluate_promotion_gate(
    CANDIDATE.candidate_id,
    measured_capability_evidence,
    lineage_valid=True,
    actual_suite_version=CANDIDATE.capability_suite_version,
    expected_suite_version=CAPABILITY_SUITE.version,
    rollback_target_hash=BASE_CONTRACT.weights_hash,
    policy_frozen=policy_hash_before_site_c == policy_hash_after_site_c,
)

# Assertion-backed release fixtures prove that incomplete evidence cannot pass.
passing_fixture = {
    "target_macro_f1_gain": 0.05,
    "legacy_A_macro_f1_regression": -0.02,
    "legacy_A_alignment_regression": -0.03,
    "site_C_ece_increase": 0.01,
}
gate_scenario_inputs = {
    "A_all_required_metrics_pass": (passing_fixture, CAPABILITY_SUITE.version, BASE_CONTRACT.weights_hash),
    "B_legacy_metric_fails": ({**passing_fixture, "legacy_A_macro_f1_regression": -1.0}, CAPABILITY_SUITE.version, BASE_CONTRACT.weights_hash),
    "C_alignment_metric_missing": ({key: value for key, value in passing_fixture.items() if key != "legacy_A_alignment_regression"}, CAPABILITY_SUITE.version, BASE_CONTRACT.weights_hash),
    "D_rollback_artifact_missing": (passing_fixture, CAPABILITY_SUITE.version, None),
    "E_suite_version_mismatch": (passing_fixture, "stale-suite-v0", BASE_CONTRACT.weights_hash),
}
gate_scenarios = []
for scenario, (scenario_evidence, suite_version, rollback_hash) in gate_scenario_inputs.items():
    result = evaluate_promotion_gate(
        scenario,
        scenario_evidence,
        lineage_valid=True,
        actual_suite_version=suite_version,
        expected_suite_version=CAPABILITY_SUITE.version,
        rollback_target_hash=rollback_hash,
        policy_frozen=True,
    )
    gate_scenarios.append({
        "scenario": scenario,
        "decision": result.decision,
        "check_states": {check.name: check.status for check in result.checks},
    })
gate_scenarios = pd.DataFrame(gate_scenarios)
expected_decisions = {
    "A_all_required_metrics_pass": "promote_to_shadow",
    "B_legacy_metric_fails": "reject",
    "C_alignment_metric_missing": "needs_review",
    "D_rollback_artifact_missing": "reject",
    "E_suite_version_mismatch": "reject",
}
assert gate_scenarios.set_index("scenario")["decision"].to_dict() == expected_decisions
assert gate_scenarios.loc[gate_scenarios["scenario"] == "C_alignment_metric_missing", "check_states"].iloc[0]["alignment_retention"] == "MISSING"
assert promotion_decision.authorization == "none"
promotion_decision, gate_scenarios


## 15. Export evidence, not a victory label

The artifact keeps local measurements separate from optional-model observations and unresolved production assumptions. It contains no hidden reasoning, secrets, raw personal data, or claim of foundation-model quality.


In [ ]:
failure_attribution = pd.DataFrame([
    {"stage": "shift_characterization", "signal": "target unchanged after adaptation", "response": "revisit data and contract"},
    {"stage": "optimization", "signal": "low train loss but weak Site B", "response": "reduce capacity or improve labels"},
    {"stage": "legacy_retention", "signal": "negative Site A regression", "response": "replay, isolation, regularization, or reject"},
    {"stage": "alignment", "signal": "classification stable but retrieval drops", "response": "constrain tower/projector movement"},
    {"stage": "routing", "signal": "adapter exists but wrong route selected", "response": "trusted context or abstain"},
    {"stage": "lineage", "signal": "base/adapter digest mismatch", "response": "block load"},
    {"stage": "release", "signal": "rollback target unavailable", "response": "stop promotion"},
])

evidence = {
    "course": "Advanced 06 — Multimodal Adaptation & Continual Learning",
    "evidence_version": "1",
    "environment": environment,
    "base_model_contract": asdict(BASE_CONTRACT),
    "capability_suite": asdict(CAPABILITY_SUITE),
    "metric_specs": {name: asdict(spec) for name, spec in METRIC_SPECS.items()},
    "metric_direction_assertions": metric_direction_assertions.to_dict(orient="records"),
    "shift_contracts": [asdict(item) for item in SHIFTS],
    "split_manifest": split_manifest.to_dict(orient="records"),
    "immutable_baseline_hash": immutable_baseline_hash,
    "base_metrics": base_metrics.to_dict(orient="records"),
    "lora_rank_sweep": rank_sweep.to_dict(orient="records"),
    "method_comparison": method_summary.to_dict(orient="records"),
    "representation_drift": representation_drift.to_dict(orient="records"),
    "stability_plasticity_controls": stability_plasticity_controls.to_dict(orient="records"),
    "adaptation_loss_method_comparison": pretext_vs_downstream.to_dict(orient="records"),
    "low_loss_capability_comparison": low_loss_candidates.to_dict(orient="records"),
    "development_policy": development_policy,
    "frozen_policy_hash": FROZEN_POLICY_HASH,
    "selected_method": SELECTED_METHOD,
    "site_c_reporting_only": site_c_report.to_dict(orient="records"),
    "continual_forgetting_matrix": forgetting_matrix.reset_index().to_dict(orient="records"),
    "continual_strategy_summary": continual_summary.to_dict(orient="records"),
    "continual_metric_evidence": continual_metric_evidence.to_dict(orient="records"),
    "replay_size_sweep": replay_size_sweep.to_dict(orient="records"),
    "replay_buffer_contract": asdict(REPLAY_BUFFER_CONTRACT),
    "replay_selection_audit": replay_selection_audit.to_dict(orient="records"),
    "isolated_adapter_routing": isolated_routing.to_dict(orient="records"),
    "routing_summary": {"routing_accuracy": routing_accuracy, "unknown_domain_abstention": unknown_domain_abstention},
    "candidate_artifact": asdict(CANDIDATE),
    "wrong_base_blocked": wrong_base_blocked,
    "release_policy": release_policy,
    "measured_capability_evidence": measured_capability_evidence,
    "promotion_decision": asdict(promotion_decision),
    "promotion_gate_scenarios": gate_scenarios.to_dict(orient="records"),
    "failure_attribution": failure_attribution.to_dict(orient="records"),
    "locally_measured_evidence": True,
    "local_proxy": {"engine": "local_tiny_dual_encoder", "foundation_model": False},
    "optional_tool_manifests": OPTIONAL_TOOL_MANIFESTS,
    "optional_model_observations": [],
    "unresolved_production_assumptions": [
        "real data rights, representativeness, privacy, and retention",
        "foundation checkpoint and processor compatibility",
        "target-hardware memory, throughput, and numerical parity",
        "multi-site repeated-run uncertainty and subgroup evidence",
        "registry signing, separation of duties, canary monitoring, and rollback drill",
    ],
    "authorization": "none",
}

evidence_path = ARTIFACT_DIR / "multimodal_adaptation_evidence.json"
decision_path = ARTIFACT_DIR / "multimodal_adaptation_decision.csv"
evidence_path.write_text(json.dumps(evidence, indent=2, default=str) + "\n", encoding="utf-8")
pd.DataFrame([{
    "candidate_id": promotion_decision.candidate_id,
    "decision": promotion_decision.decision,
    "authorization": promotion_decision.authorization,
    "rollback_target_hash": promotion_decision.rollback_target_hash,
    "frozen_policy_hash": FROZEN_POLICY_HASH,
}]).to_csv(decision_path, index=False)

assert json.loads(evidence_path.read_text())["authorization"] == "none"
assert json.loads(evidence_path.read_text())["local_proxy"]["foundation_model"] is False
evidence_path, decision_path


## 16. Production upgrade and exercises

| Teaching path | Production requirement |
| --- | --- |
| synthetic vectors | consented, licensed, source-versioned multimodal data |
| tiny dual encoder | pinned checkpoint, processor, tokenizer, and adapter runtime |
| one deterministic seed | repeated runs, uncertainty, subgroup and incident slices |
| in-memory replay | encrypted store, retention/deletion, selection audit, access control |
| trusted site string | authenticated deployment context or separately evaluated router |
| local files | signed registry artifacts, SBOM, approval separation, rollback drill |

Exercises:

1. Move LoRA from the image projection to the text projection and compare bidirectional retrieval.
2. Add source-balanced reservoir replay and prove its manifest changes when membership changes.
3. Inject a new vocabulary without visual shift and isolate language from image regression.
4. Add an unknown-domain adapter route that abstains instead of silently falling back.
5. Replace point estimates with repeated-seed mean and variability while keeping Site C reporting-only.
6. Design a single-use shadow-promotion receipt bound to candidate digest, suite version, policy, approver, expiry, and rollback target.

### Final checkpoint

You should now be able to explain why target improvement, parameter efficiency, low loss, small representation drift, or a green training job is individually insufficient evidence for safe capability evolution.
